# NUST Bank RAG Pipeline

This notebook runs the full Retrieval-Augmented Generation pipeline — from raw Excel data to answering banking product questions using **Qwen 3.5** and a **Milvus Lite** vector store.

## Run order
| # | Cell | What it does |
|---|------|--------------|
| 1 | Environment check | Detects local vs Colab |
| 2 | Install Ollama | **Colab only** — installs Ollama & starts server |
| 3 | Pull Qwen model | **Colab only** — downloads qwen3.5:4b (~2.6 GB) |
| 4 | Install packages | pip installs all Python dependencies |
| 5 | Upload files | **Colab only** — upload Excel + JSON + .py files |
| 6 | Create folders | Makes `data/` directory |
| 7 | Extract Q&A | Parses Excel → `all_qa_pairs.json` |
| 8 | Embedder stage 1 | Encodes chunks → FAISS index + metadata |
| 9 | Embedder stage 2 | Builds Milvus Lite DB → `data/milvus_bank.db` |
| 10 | RAG query | Asks a hardcoded question via Qwen 3.5 |
| 11 | Interactive query | Ask your own questions |
| 12 | Save outputs | **Colab only** — download generated files |

---
## Cell 1 — Environment Detection
Detects whether the notebook is running on **Google Colab** or **locally**.  
All subsequent cells use the `ON_COLAB` flag to skip steps that are not needed locally.

In [ ]:
import sys
import os

ON_COLAB = "google.colab" in sys.modules

if ON_COLAB:
    print("Running on Google Colab")
else:
    print("Running locally")
    print(f"Python: {sys.version}")
    print(f"Working directory: {os.getcwd()}")

---
## Cell 2 — [COLAB ONLY] Install Ollama and Start Server
> **Skip this cell if running locally.** Locally you should have Ollama installed already (`ollama serve` in a separate terminal).

This cell:
1. Downloads and installs Ollama on the Colab VM
2. Starts the Ollama server as a background process
3. Waits for it to become ready

In [ ]:
if ON_COLAB:
    import subprocess
    import time

    # Install Ollama
    print("Installing Ollama...")
    os.system("curl -fsSL https://ollama.com/install.sh | sh")

    # Start the Ollama server in the background
    print("Starting Ollama server...")
    ollama_proc = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    # Give it time to initialise
    time.sleep(8)
    print("Ollama server is up (PID", ollama_proc.pid, ")")
else:
    print("Local run — make sure 'ollama serve' is running in a separate terminal.")

---
## Cell 3 — [COLAB ONLY] Pull the Qwen 3.5 Model
> **Skip this cell if running locally.** Run `ollama pull qwen3.5:4b` in your terminal instead.

Downloads ~2.6 GB. This takes a few minutes on Colab the first time — subsequent runs (same session) skip the download.

In [ ]:
if ON_COLAB:
    print("Pulling qwen3.5:4b model (this may take several minutes)...")
    ret = os.system("ollama pull qwen3.5:4b")
    if ret == 0:
        print("Model ready.")
    else:
        print("Pull may have failed — check Ollama is running (Cell 2).")
else:
    print("Local run — ensure qwen3.5:4b is already pulled (`ollama list`).")

---
## Cell 4 — Install Python Packages
Installs all required packages from `requirements.txt`.  
- **Colab:** runs every time (fresh VM each session)
- **Local:** you already did `pip install -r requirements.txt`, so this is safe to skip

In [ ]:
if ON_COLAB:
    # Install from requirements.txt if it is present, otherwise install manually
    if os.path.exists("requirements.txt"):
        os.system("pip install -q -r requirements.txt")
    else:
        os.system(
            "pip install -q "
            "langchain-classic==1.0.1 langchain-core==1.2.17 "
            "langchain-community==0.4.1 langchain-ollama==1.0.1 "
            "langchain-huggingface==1.2.1 langchain-text-splitters==1.1.1 "
            "'pymilvus>=2.4.0' 'langchain-milvus>=0.1.0' "
            "faiss-cpu==1.13.2 sentence-transformers==5.2.3 "
            "huggingface_hub==1.5.0 ollama==0.6.1 openpyxl"
        )
    print("Packages installed.")
else:
    print("Local run — skipping install (conda venv already set up).")

---
## Cell 5 — [COLAB ONLY] Upload Project Files
> **Skip this cell if running locally.**

Upload the files that Colab needs. A file picker will appear.  
You need to upload:
- `NUST Bank-Product-Knowledge.xlsx`
- `funds_transer_app_features_faq.json`
- `embedder.py`
- `embedder_2.py`
- `llm.py`
- `search.py`
- `data/format_for_finetuning.py`
- `requirements.txt`

In [ ]:
if ON_COLAB:
    from google.colab import files as colab_files

    print("A file picker will open. Select all the files listed above.")
    uploaded = colab_files.upload()
    print("\nUploaded files:")
    for name in uploaded:
        print(f"  {name}")
else:
    print("Local run — files are already on disk.")

---
## Cell 6 — Create Required Directories

In [ ]:
os.makedirs("data", exist_ok=True)
print("Directory 'data/' is ready.")
print(f"Current directory contents: {os.listdir('.')}")

---
## Cell 7 — Step 1: Extract Q&A Pairs from Excel
Runs `data/format_for_finetuning.py`.

**Input:** `NUST Bank-Product-Knowledge.xlsx` + `funds_transer_app_features_faq.json`  
**Output:** `data/all_qa_pairs.json`, `data/finetuning_data.jsonl`, `data/finetuning_data_chat.jsonl`

In [ ]:
%run data/format_for_finetuning.py

---
## Cell 8 — Step 2: Build Stage-1 Embeddings (FAISS + Chunk Metadata)
Runs `embedder.py`.

**Input:** `data/all_qa_pairs.json`  
**Output:** `data/faiss_index.bin` (raw index), `data/chunk_metadata.json` (used by next step)  

> Downloads `all-MiniLM-L6-v2` (~80 MB) on first run.

In [ ]:
%run embedder.py

---
## Cell 9 — Step 3: Build Milvus Lite Vector Store
Runs `embedder_2.py`.

**Input:** `data/chunk_metadata.json`  
**Output:** `data/milvus_bank.db` — the Milvus Lite database (single file, no server needed)

In [ ]:
%run embedder_2.py

---
## Cell 10 — Step 4: RAG Query (hardcoded test question)
Runs `llm.py` which asks `"how do i open an account?"` through the full RAG chain.

- Connects to the Milvus Lite DB
- Retrieves top-3 matching chunks
- Sends them as context to **Qwen 3.5** running via Ollama
- Prints the generated answer

> Make sure Ollama is running with `qwen3.5:4b` pulled before executing this cell.

In [ ]:
%run llm.py

---
## Cell 11 — Interactive: Ask Your Own Question
Type any question about bank products and get an answer from the RAG chain.  
Change `YOUR_QUESTION` below to anything you want.

In [ ]:
from langchain_ollama import OllamaLLM
from langchain_milvus import Milvus
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

# ── config ──────────────────────────────────────────
MILVUS_DB   = "data/milvus_bank.db"
COLLECTION  = "bank_knowledge"
EMBED_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL   = "qwen3.5:4b"
TOP_K       = 3

# ── build chain ─────────────────────────────────────
print("Loading embeddings and Milvus...")
embed_fn = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
vec_db   = Milvus(
    embed_fn,
    connection_args={"uri": MILVUS_DB},
    collection_name=COLLECTION,
)
llm = OllamaLLM(model=LLM_MODEL)
chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vec_db.as_retriever(search_kwargs={"k": TOP_K}),
)
print("Chain ready.\n")

# ── ask a question ───────────────────────────────────
YOUR_QUESTION = "What is a Little Champs account?"

result = chain.invoke({"query": YOUR_QUESTION})
print("Question:", YOUR_QUESTION)
print("\nAnswer:", result["result"])

---
## Cell 12 — Interactive: Vector Similarity Search (No LLM)
Searches the Milvus vector store directly — no Qwen / Ollama needed.  
Useful for checking what the retriever actually finds before sending to the LLM.

In [ ]:
from langchain_milvus import Milvus
from langchain_huggingface import HuggingFaceEmbeddings

embed_fn = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vec_db   = Milvus(
    embed_fn,
    connection_args={"uri": "data/milvus_bank.db"},
    collection_name="bank_knowledge",
)

SEARCH_QUERY = "minimum balance requirement"

hits = vec_db.similarity_search_with_score(SEARCH_QUERY, k=3)

print(f"Top 3 results for: '{SEARCH_QUERY}'\n")
for rank, (doc, score) in enumerate(hits, start=1):
    meta = doc.metadata
    print(f"--- Result {rank}  (score: {score:.4f}) ---")
    print(f"  Product : {meta.get('product', 'N/A')}")
    print(f"  Q : {meta.get('question', '')}")
    print(f"  A : {meta.get('answer', '')[:200]}")
    print()

---
## Cell 13 — [COLAB ONLY] Save Generated Files to Your Computer
> **Skip this cell if running locally.** Files are already on your disk.

Downloads the important output files from Colab to your local machine before the session ends.

In [ ]:
if ON_COLAB:
    from google.colab import files as colab_files

    to_download = [
        "data/milvus_bank.db",
        "data/all_qa_pairs.json",
        "data/chunk_metadata.json",
        "data/faiss_index.bin",
    ]

    for path in to_download:
        if os.path.exists(path):
            colab_files.download(path)
            print(f"Downloaded: {path}")
        else:
            print(f"Not found (skipped): {path}")
else:
    print("Local run — files are already saved at:")
    for f in ["data/milvus_bank.db", "data/all_qa_pairs.json", "data/chunk_metadata.json"]:
        exists = os.path.exists(f)
        print(f"  {'OK' if exists else 'MISSING'}  {f}")